# DEAI – Classificatie Ames Housing

In dit notebook worden twee decision tree-classificatiemodellen gemaakt:

1. **Binaire classificatie**: voorspellen of een huis een garage heeft (`garage`)
2. **Multi-class classificatie**: voorspellen van het kwaliteitsniveau (`overallqual`)

Per model worden:
- features gekozen
- categorische features one-hot encoded
- data horizontaal en verticaal gesplitst
- een decision tree getraind
- evaluatiemetrieken berekend
- meerdere experimenten uitgevoerd


## 1. Imports


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)


## 2. Data inladen

Lees het tabblad **AmesHousing** uit het Excelbestand in.
Pas indien nodig het pad hieronder aan.


In [ ]:
file_path = "AmesHousing.xlsx"
df = pd.read_excel(file_path, sheet_name="AmesHousing")

df.head()


In [ ]:
df.info()


In [ ]:
df.columns


## 3. Kolomnamen opschonen

Om fouten met spaties in kolomnamen te voorkomen, worden de kolomnamen opgeschoond.


In [ ]:
df.columns = df.columns.str.replace(" ", "").str.lower()
df.columns


## 4. Overzicht van de beschikbare kolommen


In [ ]:
for col in df.columns:
    print(col)


# MODEL 1 – Binaire classificatie: Garage

Target:
- `garage`

Initiële featurekeuze:
- `saleprice`
- `grlivarea`
- `neighborhood`

Motivatie:
- `saleprice` hangt waarschijnlijk samen met de aanwezigheid van een garage
- `grlivarea` zegt iets over de grootte van het huis
- `neighborhood` is categorisch en locatie kan voorspellend zijn


## 5. Data voorbereiden voor model 1
- categorische feature one-hot encoden
- target apart zetten


In [ ]:
features_garage = ["saleprice", "grlivarea", "neighborhood"]
target_garage = "garage"

df_garage = df[features_garage + [target_garage]].copy()
df_garage_encoded = pd.get_dummies(df_garage, columns=["neighborhood"])

df_garage_encoded.head()


## 6. Dataset verticaal en horizontaal splitsen voor model 1

Verticaal:
- `X` = features
- `y` = target

Horizontaal:
- trainset
- testset

Hierdoor ontstaan vier stukken:
- `X_train_g`
- `X_test_g`
- `y_train_g`
- `y_test_g`


In [ ]:
X_garage = df_garage_encoded.drop(columns=[target_garage])
y_garage = df_garage_encoded[target_garage]

X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_garage,
    y_garage,
    test_size=0.15,
    random_state=42
)

print("X_train_g:", X_train_g.shape)
print("X_test_g :", X_test_g.shape)
print("y_train_g:", y_train_g.shape)
print("y_test_g :", y_test_g.shape)


## 7. Hyperparameters van DecisionTreeClassifier bekijken


In [ ]:
help(DecisionTreeClassifier)


## 8. Eerste decision tree voor model 1 trainen

Gebruikte hyperparameters:
- `max_depth=4`
- `min_samples_split=10`
- `random_state=42`


In [ ]:
tree_garage_1 = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=10,
    random_state=42
)

tree_garage_1.fit(X_train_g, y_train_g)


## 9. Evaluatie van model 1


In [ ]:
y_pred_g = tree_garage_1.predict(X_test_g)

acc_g = accuracy_score(y_test_g, y_pred_g)
prec_g = precision_score(y_test_g, y_pred_g, pos_label="yes", zero_division=0)
rec_g = recall_score(y_test_g, y_pred_g, pos_label="yes", zero_division=0)
cm_g = confusion_matrix(y_test_g, y_pred_g)

print("Accuracy :", acc_g)
print("Precision:", prec_g)
print("Recall   :", rec_g)
print("Confusion matrix:")
print(cm_g)
print()
print(classification_report(y_test_g, y_pred_g))


## 10. Visualisatie van model 1


In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(
    tree_garage_1,
    feature_names=X_garage.columns,
    class_names=["Geen garage", "Wel garage"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()


## 11. Eerste interpretatie model 1


In [ ]:
results_garage_1 = pd.DataFrame({
    "feature": X_garage.columns,
    "importance": tree_garage_1.feature_importances_
}).sort_values("importance", ascending=False)

results_garage_1.head(10)


# MODEL 2 – Multi-class classificatie: Overall Qual

Target:
- `overallqual`

Initiële featurekeuze:
- `saleprice`
- `grlivarea`
- `housestyle`

Motivatie:
- `saleprice` hangt waarschijnlijk samen met kwaliteitsniveau
- `grlivarea` zegt iets over grootte en kwaliteit
- `housestyle` is categorisch en kan invloed hebben op kwaliteitsklasse


## 12. Data voorbereiden voor model 2


In [ ]:
features_qual = ["saleprice", "grlivarea", "housestyle"]
target_qual = "overallqual"

df_qual = df[features_qual + [target_qual]].copy()
df_qual_encoded = pd.get_dummies(df_qual, columns=["housestyle"])

df_qual_encoded.head()


## 13. Dataset splitsen voor model 2


In [ ]:
X_qual = df_qual_encoded.drop(columns=[target_qual])
y_qual = df_qual_encoded[target_qual]

X_train_q, X_test_q, y_train_q, y_test_q = train_test_split(
    X_qual,
    y_qual,
    test_size=0.15,
    random_state=42
)

print("X_train_q:", X_train_q.shape)
print("X_test_q :", X_test_q.shape)
print("y_train_q:", y_train_q.shape)
print("y_test_q :", y_test_q.shape)


## 14. Eerste decision tree voor model 2 trainen

Gebruikte hyperparameters:
- `max_depth=5`
- `min_samples_split=12`
- `random_state=42`


In [ ]:
tree_qual_1 = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=12,
    random_state=42
)

tree_qual_1.fit(X_train_q, y_train_q)


## 15. Evaluatie van model 2


In [ ]:
y_pred_q = tree_qual_1.predict(X_test_q)

acc_q = accuracy_score(y_test_q, y_pred_q)
prec_q = precision_score(y_test_q, y_pred_q, average="weighted", zero_division=0)
rec_q = recall_score(y_test_q, y_pred_q, average="weighted", zero_division=0)
cm_q = confusion_matrix(y_test_q, y_pred_q)

print("Accuracy :", acc_q)
print("Precision:", prec_q)
print("Recall   :", rec_q)
print("Confusion matrix:")
print(cm_q)
print()
print(classification_report(y_test_q, y_pred_q, zero_division=0))


## 16. Visualisatie van model 2


In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    tree_qual_1,
    feature_names=X_qual.columns,
    filled=True,
    rounded=True,
    fontsize=7
)
plt.show()


## 17. Eerste interpretatie model 2


In [ ]:
results_qual_1 = pd.DataFrame({
    "feature": X_qual.columns,
    "importance": tree_qual_1.feature_importances_
}).sort_values("importance", ascending=False)

results_qual_1.head(10)


# 18. Experimenten – Model 1 (Garage)

Volgens de opdracht moeten eerdere experimenten behouden blijven.
Hieronder volgen extra configuraties met andere features en hyperparameters.


## Experiment Garage 2
Toegevoegde features:
- `yearbuilt`
- `housestyle`

Nieuwe hyperparameters:
- `max_depth=6`
- `min_samples_split=20`
- `min_samples_leaf=5`


In [ ]:
features_garage_2 = ["saleprice", "grlivarea", "yearbuilt", "neighborhood", "housestyle"]
target_garage = "garage"

df_g2 = df[features_garage_2 + [target_garage]].copy()
df_g2 = pd.get_dummies(df_g2, columns=["neighborhood", "housestyle"])

X_g2 = df_g2.drop(columns=[target_garage])
y_g2 = df_g2[target_garage]

X_train_g2, X_test_g2, y_train_g2, y_test_g2 = train_test_split(
    X_g2,
    y_g2,
    test_size=0.15,
    random_state=42
)

tree_garage_2 = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=20,
    min_samples_leaf=5,
    random_state=42
)

tree_garage_2.fit(X_train_g2, y_train_g2)
y_pred_g2 = tree_garage_2.predict(X_test_g2)

acc_g2 = accuracy_score(y_test_g2, y_pred_g2)
prec_g2 = precision_score(y_test_g2, y_pred_g2, pos_label="yes", zero_division=0)
rec_g2 = recall_score(y_test_g2, y_pred_g2, pos_label="yes", zero_division=0)

print("Accuracy :", acc_g2)
print("Precision:", prec_g2)
print("Recall   :", rec_g2)
print()
print(classification_report(y_test_g2, y_pred_g2, zero_division=0))


## Experiment Garage 3
Andere featurecombinatie:
- `saleprice`
- `overallqual`
- `yearbuilt`
- `neighborhood`

Andere hyperparameters:
- `max_depth=8`
- `min_samples_split=12`
- `min_samples_leaf=3`


In [ ]:
features_garage_3 = ["saleprice", "overallqual", "yearbuilt", "neighborhood"]
target_garage = "garage"

df_g3 = df[features_garage_3 + [target_garage]].copy()
df_g3 = pd.get_dummies(df_g3, columns=["neighborhood"])

X_g3 = df_g3.drop(columns=[target_garage])
y_g3 = df_g3[target_garage]

X_train_g3, X_test_g3, y_train_g3, y_test_g3 = train_test_split(
    X_g3,
    y_g3,
    test_size=0.15,
    random_state=42
)

tree_garage_3 = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=12,
    min_samples_leaf=3,
    random_state=42
)

tree_garage_3.fit(X_train_g3, y_train_g3)
y_pred_g3 = tree_garage_3.predict(X_test_g3)

acc_g3 = accuracy_score(y_test_g3, y_pred_g3)
prec_g3 = precision_score(y_test_g3, y_pred_g3, pos_label="yes", zero_division=0)
rec_g3 = recall_score(y_test_g3, y_pred_g3, pos_label="yes", zero_division=0)

print("Accuracy :", acc_g3)
print("Precision:", prec_g3)
print("Recall   :", rec_g3)
print()
print(classification_report(y_test_g3, y_pred_g3, zero_division=0))


# 19. Experimenten – Model 2 (Overall Qual)


## Experiment OverallQual 2
Toegevoegde features:
- `yearbuilt`
- `lotarea`
- `neighborhood`

Nieuwe hyperparameters:
- `max_depth=7`
- `min_samples_split=15`
- `min_samples_leaf=4`


In [ ]:
features_q2 = ["saleprice", "grlivarea", "yearbuilt", "lotarea", "housestyle", "neighborhood"]
target_qual = "overallqual"

df_q2 = df[features_q2 + [target_qual]].copy()
df_q2 = pd.get_dummies(df_q2, columns=["housestyle", "neighborhood"])

X_q2 = df_q2.drop(columns=[target_qual])
y_q2 = df_q2[target_qual]

X_train_q2, X_test_q2, y_train_q2, y_test_q2 = train_test_split(
    X_q2,
    y_q2,
    test_size=0.15,
    random_state=42
)

tree_q2 = DecisionTreeClassifier(
    max_depth=7,
    min_samples_split=15,
    min_samples_leaf=4,
    random_state=42
)

tree_q2.fit(X_train_q2, y_train_q2)
y_pred_q2 = tree_q2.predict(X_test_q2)

acc_q2 = accuracy_score(y_test_q2, y_pred_q2)
prec_q2 = precision_score(y_test_q2, y_pred_q2, average="weighted", zero_division=0)
rec_q2 = recall_score(y_test_q2, y_pred_q2, average="weighted", zero_division=0)

print("Accuracy :", acc_q2)
print("Precision:", prec_q2)
print("Recall   :", rec_q2)
print()
print(classification_report(y_test_q2, y_pred_q2, zero_division=0))


## Experiment OverallQual 3
Andere featurecombinatie:
- `saleprice`
- `grlivarea`
- `totalbsmtsf`
- `fullbath`
- `bedroomabvgr`
- `housestyle`

Andere hyperparameters:
- `max_depth=9`
- `min_samples_split=20`
- `min_samples_leaf=5`


In [ ]:
features_q3 = ["saleprice", "grlivarea", "totalbsmtsf", "fullbath", "bedroomabvgr", "housestyle"]
target_qual = "overallqual"

df_q3 = df[features_q3 + [target_qual]].copy()
df_q3 = pd.get_dummies(df_q3, columns=["housestyle"])

X_q3 = df_q3.drop(columns=[target_qual])
y_q3 = df_q3[target_qual]

X_train_q3, X_test_q3, y_train_q3, y_test_q3 = train_test_split(
    X_q3,
    y_q3,
    test_size=0.15,
    random_state=42
)

tree_q3 = DecisionTreeClassifier(
    max_depth=9,
    min_samples_split=20,
    min_samples_leaf=5,
    random_state=42
)

tree_q3.fit(X_train_q3, y_train_q3)
y_pred_q3 = tree_q3.predict(X_test_q3)

acc_q3 = accuracy_score(y_test_q3, y_pred_q3)
prec_q3 = precision_score(y_test_q3, y_pred_q3, average="weighted", zero_division=0)
rec_q3 = recall_score(y_test_q3, y_pred_q3, average="weighted", zero_division=0)

print("Accuracy :", acc_q3)
print("Precision:", prec_q3)
print("Recall   :", rec_q3)
print()
print(classification_report(y_test_q3, y_pred_q3, zero_division=0))


## 20. Resultaten vergelijken


In [ ]:
results = pd.DataFrame([
    ["Garage - initieel", acc_g,  prec_g,  rec_g],
    ["Garage - experiment 2", acc_g2, prec_g2, rec_g2],
    ["Garage - experiment 3", acc_g3, prec_g3, rec_g3],
    ["OverallQual - initieel", acc_q,  prec_q,  rec_q],
    ["OverallQual - experiment 2", acc_q2, prec_q2, rec_q2],
    ["OverallQual - experiment 3", acc_q3, prec_q3, rec_q3],
], columns=["model", "accuracy", "precision", "recall"])

results.sort_values("accuracy", ascending=False)


## 21. Beste configuratie per model bepalen

Kies op basis van de tabel hierboven:
- het beste garage-model
- het beste overallqual-model

Kijk niet alleen naar accuracy, maar ook naar precision en recall.


# 22. Conclusie

Beschrijf hier in eigen woorden:
- welke experimenten zijn uitgevoerd
- welke configuraties beter presteerden
- welke nieuwe experimenten geïnspireerd werden door eerdere resultaten
- hoe de gekozen hyperparameters en features de prestaties beïnvloedden
